# Statistics — Step 3 Python Walkthrough (Chapters 1–9)
This notebook mirrors the chapter sequence from *Statistics, Updated Edition (13th ed.)* by **McClave & Sincich**.

**What you get:**
- Minimal but meaningful markdown explanations
- Ready-to-run Python code blocks
- Simulations for CLT, confidence intervals, and hypothesis testing

> Tip: Run top-to-bottom. Re-run simulation cells to see sampling variability.

## Setup
Install/import common libraries used across chapters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Optional (used for two-proportion z-test in Chapter 9)
try:
    import statsmodels.stats.proportion as smp
except Exception as e:
    smp = None
    print('statsmodels not available; two-proportion z-test cell will be skipped if needed.')

np.random.seed(42)

---
# Chapter 1 — Statistics, Data, and Statistical Thinking
Python here helps illustrate the difference between **population** and **sample**, and why statistics vary from sample to sample.

### Population vs Sample (sampling variability)

In [ ]:
population = np.random.normal(loc=70, scale=10, size=100000)

# Take one sample
sample = np.random.choice(population, size=30, replace=False)

pop_mean = population.mean()
samp_mean = sample.mean()
pop_sd = population.std(ddof=0)
samp_sd = sample.std(ddof=1)

pop_mean, samp_mean, pop_sd, samp_sd

### Sampling distribution of the sample mean (preview of Chapter 6)
Even though the population is fixed, the statistic **x̄** changes across samples.

In [ ]:
sample_means = [np.mean(np.random.choice(population, size=30, replace=False)) for _ in range(2000)]

plt.hist(sample_means, bins=30)
plt.title('Sampling Distribution of the Mean (n=30)')
plt.xlabel('Sample mean')
plt.ylabel('Frequency')
plt.show()

np.mean(sample_means), np.std(sample_means)

---
# Chapter 2 — Methods for Describing Sets of Data
Descriptive statistics and graphs: histograms, mean/median, spread, skewness, and outliers.

### Histogram (quantitative data)

In [ ]:
scores = np.random.normal(70, 10, 200)
plt.hist(scores, bins=12)
plt.title('Histogram of Scores (Quantitative)')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.show()

### Bar chart (qualitative data)

In [ ]:
categories = pd.Series(['A','B','A','C','B','A','D','C','B','A','A','D'])
freq = categories.value_counts()

plt.bar(freq.index, freq.values)
plt.title('Bar Chart (Qualitative)')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

freq

### Mean vs Median with an outlier (right skew)

In [ ]:
data = np.array([10, 12, 13, 14, 15, 16, 100])
data.mean(), np.median(data), data.std(ddof=1)

### Right-skewed distribution visualization (exponential)

In [ ]:
right_skewed = np.random.exponential(scale=2, size=1000)
plt.hist(right_skewed, bins=30)
plt.title('Right-Skewed Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

np.mean(right_skewed), np.median(right_skewed)

### Outliers via IQR rule (boxplot + quartiles)

In [ ]:
plt.boxplot(right_skewed)
plt.title('Boxplot')
plt.show()

q1, q3 = np.percentile(right_skewed, [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

q1, q3, iqr, lower, upper

---
# Chapter 3 — Probability
Use Python to enumerate sample spaces, compute conditional probability ideas, and demonstrate Bayes' rule.

### Sample space for coin tosses

In [ ]:
from itertools import product

outcomes = list(product(['H','T'], repeat=3))
outcomes[:10], len(outcomes)

### Conditional probability (simple numeric example)
If P(A)=0.6 and P(B|A)=0.5, then P(A∩B)=P(A)P(B|A).

In [ ]:
P_A = 0.6
P_B_given_A = 0.5
P_A_and_B = P_A * P_B_given_A
P_A_and_B

### Bayes' rule (medical test intuition)
Shows why base rates matter.

In [ ]:
P_D = 0.01                 # prevalence
P_pos_given_D = 0.99       # sensitivity
P_pos_given_notD = 0.05    # false positive rate

P_pos = P_pos_given_D*P_D + P_pos_given_notD*(1-P_D)
P_D_given_pos = (P_pos_given_D*P_D) / P_pos

P_pos, P_D_given_pos

---
# Chapter 4 — Discrete Random Variables
Binomial, Poisson, Hypergeometric using SciPy.

### Binomial: P(X=3) where X~Bin(n=10,p=0.4)

In [ ]:
from scipy.stats import binom

binom.pmf(3, n=10, p=0.4)

### Binomial mean and variance

In [ ]:
binom.mean(n=10, p=0.4), binom.var(n=10, p=0.4)

### Poisson: P(X=2) where X~Pois(λ=3)

In [ ]:
from scipy.stats import poisson

poisson.pmf(2, mu=3), poisson.mean(mu=3), poisson.var(mu=3)

### Hypergeometric: N=50 total, K=5 successes, sample n=10, find P(X=1)

In [ ]:
from scipy.stats import hypergeom

hypergeom.pmf(1, N=50, K=5, n=10)

---
# Chapter 5 — Continuous Random Variables
Normal distribution probabilities, z-scores, and normal probability plots.

### Standard normal CDF examples

In [ ]:
from scipy.stats import norm

norm.cdf(1.96), norm.cdf(1) - norm.cdf(-1)  # ~97.5% below 1.96; ~68% within 1σ

### z-score calculation

In [ ]:
x, mu, sigma = 85, 70, 10
z = (x - mu) / sigma
z

### Normal probability plot (Q–Q plot)

In [ ]:
data = np.random.normal(70, 10, 200)
stats.probplot(data, plot=plt)
plt.title('Normal Probability Plot')
plt.show()

---
# Chapter 6 — Sampling Distributions
**Central Limit Theorem (CLT)** via simulation and standard error behavior.

### CLT: sampling means from a skewed population
Population is exponential (right-skewed). Sampling means become more normal as n increases.

In [ ]:
skew_pop = np.random.exponential(scale=2, size=200000)

means_n5 = [np.mean(np.random.choice(skew_pop, 5, replace=False)) for _ in range(5000)]
means_n30 = [np.mean(np.random.choice(skew_pop, 30, replace=False)) for _ in range(5000)]

plt.hist(means_n5, bins=30)
plt.title('Sampling Distribution of Mean (n=5)')
plt.xlabel('Sample mean')
plt.ylabel('Frequency')
plt.show()

plt.hist(means_n30, bins=30)
plt.title('Sampling Distribution of Mean (n=30)')
plt.xlabel('Sample mean')
plt.ylabel('Frequency')
plt.show()

np.mean(means_n5), np.std(means_n5), np.mean(means_n30), np.std(means_n30)

### Standard error shrinkage ~ 1/sqrt(n) (illustration)

In [ ]:
def simulate_se(n, reps=3000):
    means = [np.mean(np.random.choice(skew_pop, n, replace=False)) for _ in range(reps)]
    return np.std(means)

ses = {n: simulate_se(n) for n in [5, 10, 30, 50, 100]}
ses

---
# Chapter 7 — Confidence Intervals
Compute common confidence intervals for a mean (t) and a proportion (z).

### t-interval for the population mean (σ unknown)

In [ ]:
import math
from scipy.stats import t

xbar, s, n = 70, 12, 25
alpha = 0.05
tcrit = t.ppf(1 - alpha/2, df=n-1)
ci = (xbar - tcrit*s/math.sqrt(n), xbar + tcrit*s/math.sqrt(n))
tcrit, ci

### z-interval for a population proportion (large-sample)

In [ ]:
phat, n = 0.30, 400
zcrit = norm.ppf(0.975)
se = math.sqrt(phat*(1-phat)/n)
ci_p = (phat - zcrit*se, phat + zcrit*se)
zcrit, ci_p

---
# Chapter 8 — Hypothesis Testing
One-sample tests (mean), p-values, and decisions.

### One-sample t-test (mean)

In [ ]:
from scipy.stats import ttest_1samp

sample = np.random.normal(72, 10, 30)
res = ttest_1samp(sample, popmean=70)
res

### Manual t-statistic + p-value (two-sided)

In [ ]:
xbar = sample.mean()
s = sample.std(ddof=1)
n = len(sample)
t_stat = (xbar - 70) / (s / math.sqrt(n))
p_val = 2*(1 - t.cdf(abs(t_stat), df=n-1))
t_stat, p_val

---
# Chapter 9 — Two-Sample Inference
Independent two-sample t-test (Welch), paired t-test, and two-proportion z-test.

### Welch two-sample t-test (independent samples, unequal variances)

In [ ]:
from scipy.stats import ttest_ind

group1 = np.random.normal(75, 10, 30)
group2 = np.random.normal(70, 12, 30)
ttest_ind(group1, group2, equal_var=False)

### Paired t-test (before vs after)

In [ ]:
from scipy.stats import ttest_rel

before = np.array([80, 82, 78, 85, 83])
after  = np.array([82, 84, 80, 86, 85])
ttest_rel(after, before)

### Two-proportion z-test (requires statsmodels)
Counts = successes in each group; nobs = total in each group.

In [ ]:
if smp is None:
    print('statsmodels not available in this environment.')
else:
    count = [120, 100]
    nobs = [400, 380]
    z_stat, p_val = smp.proportions_ztest(count, nobs)
    (z_stat, p_val)

---
## Done
You now have a single end-to-end Python notebook aligned to Chapters 1–9.
- Re-run simulation cells to see variability.
- Adjust parameters (n, p, μ, σ) to practice exam scenarios.